In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
import math
import copy
from torch import optim

import numpy as np
import pandas as pd
import pickle
from torch.utils.data import DataLoader, TensorDataset


class KANLayer(nn.Module):

  def __init__(self, in_features, out_features, num_basis=8):
    super().__init__()
    self.in_features = in_features
    self.out_features = out_features
    self.num_basis = num_basis

    centers = torch.linspace(-3.0, 3.0, num_basis).view(1, 1, num_basis)
    self.centers = nn.Parameter(centers.repeat(1, in_features, 1))

    self.log_widths = nn.Parameter(torch.zeros(1, in_features, num_basis))

    self.weight = nn.Parameter(torch.randn(in_features * num_basis, out_features) * 0.02)
    self.bias = nn.Parameter(torch.zeros(out_features))

  def forward(self, x):
    x_exp = x.unsqueeze(-1)

    widths = torch.exp(self.log_widths) + 1e-6
    basis = torch.exp(-((x_exp - self.centers) ** 2) / (2.0 * widths ** 2))

    phi = basis.reshape(x.size(0), self.in_features * self.num_basis)
    out = phi @ self.weight + self.bias
    return out


class KAN_CVAE(nn.Module):
    def __init__(self, x_dim=6, cond_dim=3, latent_dim=30, num_basis=8, dropout=0.2):
      super().__init__()

      self.x_dim = x_dim
      self.cond_dim = cond_dim
      self.latent_dim = latent_dim

      self.enc1 = KANLayer(x_dim + cond_dim, 512, num_basis=num_basis)
      self.enc2 = KANLayer(512, 256, num_basis=num_basis)
      self.enc3 = KANLayer(256, 128, num_basis=num_basis)
      self.mu_head = KANLayer(128, latent_dim, num_basis=num_basis)
      self.logvar_head = KANLayer(128, latent_dim, num_basis=num_basis)
      self.dropout = nn.Dropout(dropout)

      self.dec1 = KANLayer(latent_dim + cond_dim, 128, num_basis=num_basis)
      self.dec2 = KANLayer(128, 256, num_basis=num_basis)
      self.dec3 = KANLayer(256, 512, num_basis=num_basis)
      self.out_head = KANLayer(512, x_dim, num_basis=num_basis)

    def encode(self, x, condition):
      h = torch.cat([x, condition], dim=1)
      h = self.dropout(F.silu(self.enc1(h)))
      h = self.dropout(F.silu(self.enc2(h)))
      h = self.dropout(F.silu(self.enc3(h)))
      mu = self.mu_head(h)
      logvar = self.logvar_head(h)
      return mu, logvar

    def reparameterize(self, mu, logvar):
      std = torch.exp(0.5 * logvar)
      eps = torch.randn_like(std)
      return mu + eps * std

    def decode(self, z, condition):
      h = torch.cat([z, condition], dim=1)
      h = self.dropout(F.silu(self.dec1(h)))
      h = self.dropout(F.silu(self.dec2(h)))
      h = self.dropout(F.silu(self.dec3(h)))
      return self.out_head(h)

    def forward(self, x, condition):
      mu, logvar = self.encode(x, condition)
      z = self.reparameterize(mu, logvar)
      recon = self.decode(z, condition)
      return recon, mu, logvar


def cvae_loss(recon_x, x, mu, logvar, beta=0.001):
  mse = F.mse_loss(recon_x, x, reduction='sum')
  kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
  return mse + beta * kld


def train_kan_cvae(
    cvae,
    train_loader,
    val_loader,
    epochs=300,
    patience=50,
    beta=0.001,
    lr=1e-3,
    device="cuda"
):
  cvae.to(device)

  optimizer = optim.AdamW(cvae.parameters(), lr=lr)
  scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.7)

  best_val_loss = float("inf")
  best_state = None
  no_improve_epochs = 0

  train_losses = []
  val_losses = []

  for epoch in range(epochs):

    cvae.train()
    train_total = 0.0

    for y, x in train_loader:
      y = y.to(device)
      x = x.to(device)

      optimizer.zero_grad()

      recon_batch, mu, logvar = cvae(x, y)
      loss = cvae_loss(recon_batch, x, mu, logvar, beta=beta)

      loss.backward()
      optimizer.step()

      train_total += loss.item()

    avg_train_loss = train_total / len(train_loader.dataset)
    train_losses.append(avg_train_loss)

    cvae.eval()
    val_total = 0.0

    with torch.no_grad():
      for y, x in val_loader:
        y = y.to(device)
        x = x.to(device)

        recon_batch, mu, logvar = cvae(x, y)
        val_loss = cvae_loss(recon_batch, x, mu, logvar, beta=beta)

        val_total += val_loss.item()

    avg_val_loss = val_total / len(val_loader.dataset)
    val_losses.append(avg_val_loss)

    print(f"[KAN-CVAE] Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
      best_val_loss = avg_val_loss
      best_state = copy.deepcopy(cvae.state_dict())
      no_improve_epochs = 0
    else:
      no_improve_epochs += 1
      if no_improve_epochs >= patience:
        print(f"Early stopping at epoch {epoch+1}. No validation improvement for {patience} epochs.")
        break

    scheduler.step()

  if best_state is not None:
    cvae.load_state_dict(best_state)
    print(f"Loaded best KAN-CVAE model with Val Loss: {best_val_loss:.4f}")

  return cvae


def generate_outputs(model, input_data, num_samples=10):
  model.eval()
  device = next(model.parameters()).device  # get model device

  with torch.no_grad():
    input_df = pd.DataFrame(
    input_data,columns=['Frequency (Hz)', 'Storage modulus (Pa)', 'Loss modulus (Pa)'])
    input_data_scaled = Y_scaler.transform(input_df)

    conditions = torch.tensor(input_data_scaled, dtype=torch.float32, device=device)

    outputs = []
    for _ in range(num_samples):
      z = torch.randn(conditions.size(0), 30, device=device)
      output = model.decode(z, conditions)
      output = X_scaler.inverse_transform(output.cpu().numpy())
      outputs.append(output)

  return outputs

In [ ]:
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.set_default_dtype(torch.float32)


device = 'cuda' if torch.cuda.is_available() else "cpu"

df_train = pd.read_csv('/content/train.csv')
df_val = pd.read_csv('/content/val.csv')



X_train, X_val = df_train.iloc[:, 0:6], df_val.iloc[:, 0:6]
y_train, y_val = df_train.iloc[:, 6:9], df_val.iloc[:, 6:9]

with open('/content/X_scaler_cvae.pkl', 'rb') as f:
  X_scaler = pickle.load(f)

with open('/content/Y_scaler_cvae.pkl', 'rb') as f:
  Y_scaler = pickle.load(f)


X_train_scaled = X_scaler.transform(X_train)
y_train_scaled = Y_scaler.transform(y_train)

X_val_scaled = X_scaler.transform(X_val)
y_val_scaled = Y_scaler.transform(y_val)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)

train_data = TensorDataset(y_train_tensor, X_train_tensor)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_scaled, dtype=torch.float32)

val_data = TensorDataset(y_val_tensor, X_val_tensor)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)


model = KAN_CVAE(
    x_dim=6,
    cond_dim=3,
    latent_dim=30,
    num_basis=8,
    dropout=0.2
)

best_kan_cvae = train_kan_cvae(
    model,
    train_loader,
    val_loader,
    epochs=300,
    patience=20,
    beta=0.001,
    lr=1e-3,
    device=device
)

df_test = pd.read_csv("/content/test.csv")

frequency, storage_modulus, loss_modulus = df_test['Frequency (Hz)'], df_test['Storage modulus (Pa)'], df_test['Loss modulus (Pa)']


# Initialize an empty DataFrame with specified columns
columns = ['Acrylamide Conc. %', 'Bis-acrylamide conc %', 'Photo-initiator conc. %', 'Layer Height. (micron)',
          'Bottom Layer exposure time (s) ', 'Exposure time (s)', 'Frequency (Hz)',
          'Storage modulus (Pa)', 'Loss modulus (Pa)']

rows = []

for i, j, k in zip(frequency, storage_modulus, loss_modulus):
    input_data = [[i, j, k]]
    output_parameters  = generate_outputs(
    best_kan_cvae,
    input_data,
    num_samples=1,
)[0]
    combined_data = list(output_parameters[0]) + [i, j, k]
    rows.append(combined_data)

df = pd.DataFrame(rows, columns=columns)

df.to_csv('synth_kan_base.csv', index=False)
print("\nSaved synthetic data!")

[KAN-CVAE] Epoch 1/300, Train Loss: 7.1751, Val Loss: 6.8526
[KAN-CVAE] Epoch 2/300, Train Loss: 6.5198, Val Loss: 7.0430
[KAN-CVAE] Epoch 3/300, Train Loss: 6.4779, Val Loss: 7.4970
[KAN-CVAE] Epoch 4/300, Train Loss: 6.6171, Val Loss: 7.4014
[KAN-CVAE] Epoch 5/300, Train Loss: 6.2728, Val Loss: 6.7421
[KAN-CVAE] Epoch 6/300, Train Loss: 5.8724, Val Loss: 5.8877
[KAN-CVAE] Epoch 7/300, Train Loss: 5.5942, Val Loss: 6.9897
[KAN-CVAE] Epoch 8/300, Train Loss: 5.7461, Val Loss: 6.8147
[KAN-CVAE] Epoch 9/300, Train Loss: 5.4551, Val Loss: 5.9183
[KAN-CVAE] Epoch 10/300, Train Loss: 5.5158, Val Loss: 5.4891
[KAN-CVAE] Epoch 11/300, Train Loss: 5.3722, Val Loss: 5.6638
[KAN-CVAE] Epoch 12/300, Train Loss: 5.3150, Val Loss: 5.5641
[KAN-CVAE] Epoch 13/300, Train Loss: 5.3635, Val Loss: 5.9244
[KAN-CVAE] Epoch 14/300, Train Loss: 5.4657, Val Loss: 6.0700
[KAN-CVAE] Epoch 15/300, Train Loss: 5.3055, Val Loss: 5.8623
[KAN-CVAE] Epoch 16/300, Train Loss: 5.3703, Val Loss: 7.0736
[KAN-CVAE] Epoch 